# Expansion layer — Colab test

Internal engineering check: import the user-facing expansion layer and the
engine straight from this repository, then re-solve the two example models —
the Chapter 11 AK economy, and the internal-habit model from the book's
*Uncertainty Expansion — Computation Process* appendix — and verify the
numbers against their references.

**Runtime → Run all** (~1 min: one `pip install`, a shallow clone, two solves).

In [ ]:
%pip -q install autograd
import os, sys
if not os.path.isdir("QuantMFR-Colab"):
    !git clone -q --depth 1 https://github.com/as7391746/QuantMFR-Colab
sys.path.insert(0, os.path.abspath("QuantMFR-Colab/expansion"))
print("ready")

In [ ]:
# AK economy (Chapter 11), base scenario: gamma = 8.001, rho = 1.001
import numpy as np
from ak_example import ak_model, PARAMS, start_values

m = ak_model()
params = dict(PARAMS)
params.update({"gamma": 8.001, "rho": 1.001, "alpha": 0.0922 / 4})
sol = m.solve(params, start=start_values(0.0922 / 4))

curve = sol.price_elasticity(shock=1, T=160, quantile=0.5)
print("price elasticity (growth shock, median state):")
print(f"  t = 1 quarter : {curve[0]:.8f}   (reference 0.11901239)")
print(f"  t = 40 years  : {curve[-1]:.8f}   (reference 0.13483263)")
assert abs(curve[0] - 0.11901239) < 1e-6 and abs(curve[-1] - 0.13483263) < 1e-6
print("AK check: PASS")

In [ ]:
# Internal-habit model, exactly as in the book's appendix notebook
from habit_example import habit_model, PARAMS as HPARAMS, APPENDIX_SS

mh = habit_model()
solh = mh.solve(HPARAMS,
                start={"imh": 0.016, "imk": 0.076,
                       "Y": float(np.log(6.3e-6)), "X": -4.3,
                       "growth": 0.005},
                tol=1e-5)
ss = np.asarray(solh.raw["ss"], float).flatten()
diff = np.max(np.abs(ss - APPENDIX_SS))
print(f"steady state vs the appendix notebook's stored output: "
      f"max abs diff {diff:.2e}")
assert diff < 1e-6
print("habit check: PASS")

Both models go through the same `Model` declaration and the same engine.
A different model = a different declaration; the layer does the rest
(canonical naming, ordered arguments, and the steady-state starting vector
are handled internally).